In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

from sqlalchemy import create_engine
import psycopg2

import json
import os


In [2]:
with open('config.json') as config_file:
    config = json.load(config_file)

In [3]:
conn = psycopg2.connect(host = config['db_host'],
                        dbname = config['db_name'] ,
                        user = config['db_user'],
                        password = config['db_password'] ,
                        port = config['db_port'])

***

<br>

**Allow user to insert desired ticker and metric.**

Insert into market_indices table. This table is the ticker, name, and index type.

In [6]:
# Create a cursor object to interact with the database
cursor = conn.cursor()

# Data to insert for the market indices (S&P 500 and Nasdaq)
indices_data = [
    ("SPX", "S&P 500", "Equity"),        # S&P 500 index
    ("RTY", "Russell 2000", "Equity"),   # Russell 2000 index
    ("NDX", "Nasdaq 100", "Equity"),     # Nasdaq 100 index
    ("INDU", "Dow Jones", "Equity"),     # Dow Jones Industrial Average
    ("VIX", "CBOE Volatility Index", "Equity"),  # VIX
    ("DAX", "DAX", "Equity"),            # DAX (Germany stock market index)
    ("FTSE", "FTSE 100", "Equity"),      # FTSE 100 (UK stock market index)
    ("NIKKEI", "Nikkei 225", "Equity"),  # Nikkei 225 (Japan stock market index)
    ("HSI", "Hang Seng", "Equity"),      # Hang Seng Index (Hong Kong stock market index)
    ("ASX200", "ASX 200", "Equity"),    # ASX 200 (Australia stock market index)
    ("CAC40", "CAC 40", "Equity"),      # CAC 40 (France stock market index)
    ("IBEX35", "IBEX 35", "Equity"),    # IBEX 35 (Spain stock market index)
    ("SSECOM", "SSE Composite", "Equity"),  # SSE Composite Index (China stock market index)
    ("STOXX50E", "EURO STOXX 50", "Equity"),  # EURO STOXX 50 (Eurozone stock market index)
    ("Bovespa", "Bovespa", "Equity"),   # Bovespa (Brazil stock market index)
    ("TSX", "TSX Composite", "Equity"), # TSX Composite (Canada stock market index)
    ("MEXBOL", "IPC", "Equity"),        # IPC (Mexico stock market index)
    ("KOSPI", "KOSPI", "Equity"),       # KOSPI (South Korea stock market index)
    ("TSEC", "Taiwan Weighted", "Equity")  # Taiwan Weighted Index
]


# Insert data into the market_indices table
insert_query = """
    INSERT INTO market_indices (index_ticker, index_name, index_type)
    VALUES (%s, %s, %s)
    ON CONFLICT (index_ticker) DO NOTHING;  
"""

# Loop through the data and insert it into the database
for index_data in indices_data:
    cursor.execute(insert_query, index_data)

# Commit the transaction to save changes
conn.commit()

# Print success message
print("Data inserted successfully.")

# Close the cursor and connection
cursor.close()

Data inserted successfully.


Insert into index_metrics table.

In [7]:
# Create a cursor object to interact with the database
cursor = conn.cursor()

# Data to insert for the index_metrics table, including 'Close'
metrics_data = [
    ("Close", "USD", "Closing price of the index")  # Insert 'Close' as a metric
]

# Insert data into the index_metrics table
insert_query = """
    INSERT INTO index_metrics (metric_name, metric_unit, metric_description)
    VALUES (%s, %s, %s)
    ON CONFLICT (metric_name) DO NOTHING;  -- Avoid conflicts if 'Close' already exists
"""

# Loop through the data and insert it into the database
for metric_data in metrics_data:
    cursor.execute(insert_query, metric_data)

# Commit the transaction to save changes
conn.commit()

# Print success message
print("Data inserted successfully.")

# Close the cursor and connection
cursor.close()


Data inserted successfully.


<br>

***

In [8]:
stock_indexes = {
    "SPX": "^GSPC",         # S&P 500 index
    "RTY": "^RUT",          # Russell 2000 index
    "NDX": "^NDX",          # Nasdaq 100 index
    "INDU": "^DJI",         # Dow Jones Industrial Average
    "VIX": "^VIX",          # CBOE Volatility Index (VIX)
    "DAX": "^GDAXI",        # DAX (Germany stock market index)
    "FTSE": "^FTSE",        # FTSE 100 (UK stock market index)
    "NIKKEI": "^N225",      # Nikkei 225 (Japan stock market index)
    "HSI": "^HSI",          # Hang Seng Index (Hong Kong stock market index)
    "ASX200": "^AXJO",      # ASX 200 (Australia stock market index)
    "CAC40": "^FCHI",       # CAC 40 (France stock market index)
    "IBEX35": "^IBEX",      # IBEX 35 (Spain stock market index)
    "SSECOM": "^SSEC",      # SSE Composite Index (China stock market index)
    "STOXX50E": "^STOXX50E", # EURO STOXX 50 (Eurozone stock market index)
    "Bovespa": "^BVSP",     # Bovespa (Brazil stock market index)
    "TSX": "^TSX",          # TSX Composite (Canada stock market index)
    "MEXBOL": "^MXX",       # IPC (Mexico stock market index)
    "KOSPI": "^KS11",       # KOSPI (South Korea stock market index)
    "TSEC": "^TWII",        # Taiwan Weighted Index
}


In [9]:
len(stock_indexes)

19

***

**Read in the available index tickers, metric_ids, and dates to put together df showing what and when to pull**

In [10]:
def check_available():
    cursor = conn.cursor()
    cursor.execute("SELECT index_id,index_ticker FROM market_indices")
    indices = pd.DataFrame(cursor.fetchall(), columns=["index_id","index_ticker"])
    indices['yfinance_ticker'] = indices['index_ticker'].map(stock_indexes)

    cursor.execute("SELECT metric_id,metric_name FROM index_metrics")
    metrics = pd.DataFrame(cursor.fetchall(), columns=["metric_id","metric_name"])
    
    crossed = indices.merge(metrics,how='cross')
    
    cursor.execute("SELECT index_id,metric_id, max(performance_date) as max_date FROM index_performance group by 1,2;")
    max_dates = pd.DataFrame(cursor.fetchall(), columns=["index_id","metric_id","max_date"])

    combo_w_max_date = crossed.merge(max_dates,on = ['index_id','metric_id'],how='left')
    return(combo_w_max_date)

In [11]:
combos = check_available()

In [12]:
combos

,index_id,index_ticker,yfinance_ticker,metric_id,metric_name,max_date
0,5,SPX,^GSPC,1,Close,2025-02-15
1,6,NDX,^NDX,1,Close,2025-02-15
2,10,RTY,^RUT,1,Close,2025-02-15
3,12,INDU,^DJI,1,Close,2025-02-15
4,13,VIX,^VIX,1,Close,2025-02-15
5,14,DAX,^GDAXI,1,Close,2025-02-15
6,15,FTSE,^FTSE,1,Close,2025-02-15
7,16,NIKKEI,^N225,1,Close,2025-02-15
8,17,HSI,^HSI,1,Close,2025-02-15
9,18,ASX200,^AXJO,1,Close,2025-02-15


***

**Begin extracting performance from yfinance using the available index + metric combinations.**

In [17]:

def extract(row):
    ticker = row['yfinance_ticker']
    index_id = row['index_id']
    metric_id = row['metric_id']
    
    # Handle the start_date based on max_date
    if pd.isna(row['max_date']):
        start_date = '2024-01-01'
    else:
        start_date = row['max_date']
        
    end_date = pd.to_datetime('today').strftime("%Y-%m-%d")

    # Desired metric name (not used in the current code but may be used later)
    desired_metric = row['metric_name']

    # Download the data
    data = yf.download(ticker, start=start_date, end=end_date)

    # Check if data is returned
    if data.empty:
        print(f"No data found for ticker: {ticker}")
        return pd.DataFrame()  # Return an empty DataFrame if no data is found

    # Process the data
    formatted_data = data[[desired_metric]].reset_index()  # Resetting index to get 'Date' as a column
    formatted_data[desired_metric] = formatted_data[desired_metric].round(2)

    # Rename columns
    formatted_data.columns = ['performance_date', 'metric_value']
    formatted_data['index_id'] = index_id
    formatted_data['metric_id'] = metric_id

    formatted_data = formatted_data[['index_id', 'metric_id', 'performance_date', 'metric_value']]

    # Return the formatted data as a DataFrame
    return formatted_data

In [18]:
extract_res = combos.apply(extract, axis=1)
extract_res = pd.concat(extract_res.tolist(), ignore_index=True)

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['^SSEC']: YFTzMissingError('possibly delisted; no timezone found')


No data found for ticker: ^SSEC


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['^TSX']: YFTzMissingError('possibly delisted; no timezone found')
[*********************100%***********************]  1 of 1 completed

No data found for ticker: ^TSX



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


***

**Group the result above to get min and max date for each index + metric combination.**

In [19]:
def grouping(df):
    df_grouped = df.groupby(['index_id', 'metric_id'])['performance_date'].agg(['min', 'max']).reset_index().rename(columns={'min': 'min_day', 'max': 'max_day'})
    return df_grouped

***

**Get all the necessary day 15 performance for the necessary months for each index + metric combination.**

In [20]:
def start_end(row):
    # Create a date range for the 15th of each month within the range of the performance_date
    start_date = row['min_day'].replace(day=1)  # Start from the first day of the month
    if row['max_day'].day < 15:
        end_date = (row['max_day'].replace(day=1) - pd.DateOffset(months=1)).replace(day=1)
    else:
        end_date = row['max_day'].replace(day=1)  # End at the first day of the last month
    dates = pd.date_range(start=start_date, end=end_date, freq='MS') + pd.DateOffset(days=14)  # 15th of each month

    # Generate all combinations of index_id, metric_id, and the 15th of each month
    combinations = pd.MultiIndex.from_product(
        [[row['index_id']], [row['metric_id']], dates],
        names=['index_id', 'metric_id', 'performance_date']
    )
    
    # # Create a new DataFrame with all combinations of index_id, metric_id, and the 15th of each month
    df_extended = pd.DataFrame(index=combinations).reset_index()
    return(df_extended)

***

**Forward fill and return only 15ths for each month and each combination.**

In [21]:
def forward_fill_15(df1,df2):

    # Merge with original DataFrame on index_id, metric_id, and performance_date
    df_merged = pd.merge(df1, df2, on=['index_id', 'metric_id', 'performance_date'], how='outer')
    df_merged = df_merged.sort_values(by=['index_id', 'metric_id', 'performance_date'])
    
    
    # Forward fill the missing metric_value
    df_merged['metric_value'] = df_merged.groupby(['index_id', 'metric_id'])['metric_value'].ffill()
    df_merged = df_merged[df_merged['performance_date'].dt.day == 15]
    return(df_merged)

***

***

<br>

**Transform**

In [22]:
def transform(df):
    extract_grouped = grouping(df)
    req_dates = extract_grouped.apply(start_end,axis=1)
    req_dates = pd.concat(req_dates.tolist(), ignore_index=True)
    filled = forward_fill_15(df,req_dates)
    return(filled)

In [23]:
df = transform(extract_res)

***

***

**Load**

In [24]:
cur = conn.cursor()

# Define the insert query with ON CONFLICT DO NOTHING
insert_query = """
    INSERT INTO index_performance (index_id, metric_id, performance_date, metric_value)
    VALUES (%s, %s, %s, %s)
    ON CONFLICT (index_id, metric_id, performance_date) 
    DO NOTHING;
"""



In [25]:
# Insert data into the table
for index, row in df.iterrows():
    # Extract values from each row
    try:
        values = (row['index_id'], row['metric_id'], row['performance_date'], row['metric_value'])
    
    # Execute the insert query
        cur.execute(insert_query, values)
    except:
        print(f"Duplicate entry found: {row}. Skipping insert.")

# Commit the transaction
conn.commit()

# Close the cursor and connection
cur.close()
conn.close()

print('Data insert complete')

Data insert complete
